In [4]:
!pip install -q "sagemaker>=2.100.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sse-starlette 3.2.0 requires starlette>=0.49.1, but you have starlette 0.45.3 which is incompatible.
gradio 5.42.0 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.


In [ ]:
import pandas as pd
import sagemaker
import torch
import numpy as np
from torch.utils.data import DataLoader,Dataset
from transformers import DistilBertTokenizer,DistilBertModel
from tqdm import tqdm
import argparse
import pandas as pd

s3_path='s3://multi-class-text-classification-data/training/newsCorpora.csv'
df=pd.read_csv(s3_path,sep="\t",names=['ID','TITLE','URL','PUBLISHER','CATEGORY','STORY','HOSTNAME','TIMESTAMP'])

df_work=df.copy()
df_work=df_work[['TITLE','CATEGORY']]

my_dict={
    'e':'Entertainment',
    'b':'Business',
    't':'Science',
    'm':'Health'
}

def update_category(x):
    return my_dict[x]


df_work['CATEGORY']=df_work['CATEGORY'].apply(lambda x: update_category(x))

encode_dict={}
def encode_category(x):
    if x not in encode_dict.keys():
        encode_dict[x]=len(encode_dict)
    return encode_dict[x]

df_work['ENCODED']=df_work['CATEGORY'].apply(lambda x: encode_category(x))


tokenizer=DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class NewsDataset(Dataset):
    def __init__(self,dataframe,tokenizer,max_length):
        self.len=len(dataframe)
        self.data= dataframe
        self.tokenizer=tokenizer
        self.max_length=max_length

    def __getitem__(self,index):
        title=str(self.data.TITLE[index])
        title=" ".join(title.split())

        inputs= self.tokenizer.encode_plus(
                title,
                None,
                add_special_tokens= True,
                max_length=self.max_length,
                padding='max_length',
                return_token_type_ids=True,
                truncations= True,
                return_attention_mask= True
            )
        ids=inputs['input_ids']
        mask= inputs['attention_mask']

        return {
            'ids': torch.tensor(ids,dtype=torch.long),
            'mask': torch.tensor(mask,dtype=torch.long),
            'targets': torch.tensor(self.data.ENCODED[index],dtype=torch.long)
        }
    def __len__(self):
        return self.len


training_size=0.8

train_dataset=df.sample(frac=trining_size,random_state=200)
test_dataset=df.drop(train_dataset.index).reset_index(drop=True)

train_dataset=train_dataset.reset_index(drop=True)

print(f"Full Dataset {df.shape}")
print(f"Trian Dataset {train_dataset.shape}")
print(f"Test Dataset {test_dataset.shape}")

MAX_LEN=512
TRAIN_BATCH_SIZE=4
VALID_BATCH_SIZE=2

training_set=NewsDatasets(train_dataset,tokenizer,MAX_LEN)
testing_set=NewsDatasets(test_dataset,tokenizer,MAX_LEN)

train_parameters={
    'batch_size':TRAIN_BATCH_SIZE,
    'shuffle': True,
    'num_workers': 0
}

test_parameters={
    'batch_size':VALID_BATCH_SIZE,
    'shuffle': True,
    'num_workers': 0
}

training_loader=DataLoader(training_set,**train_parameters)
test_loader=DataLoader(testing_set,**test_parameters)

class DistilBertModel(torch.nn.Module):
    super().__init__()
    self.l1=DistilBertModel.from_pretrained('distilbert-base-uncased')
    self.pre_classifier=torch.nn.Linear(768,768)
    self.dropout=torch.nn.Dropout(0.3)
    self.classifier=torch.nn.Linear(768,4)

    def forward(self,input_ids,attention_mask):#Runs after every epoch
        output_1=self.l1(input_ids=input_ids,attention_mask=attention_mask)
        hidden_state=output_1[0]#returns output layer from distillbert model
        print(hidden_state)
        pooler=hidden_state[:,0]#Grabs the [CLS] tokens of each sequence
        pooler=self.pre_classifier(pooler)
        pooler=torch.nn.Relu()(pooler)
        pooler=self.dropout(pooler)
        output=self.classifier(pooler)
        return output


def calculate_accu(big_idx,target):
    n_correct=(big_idx==target).sum().item()
    return n_correct

def train(epoch,model,training_loader,optimizer,loss_function):
    tr_loss=0
    n_correct=0
    nb_tr_steps=0
    nb_tr_examples=0
    
    
    model.train()


    for _,data in enumerate(training_loader,0):
        ids=data['ids'].to(device,dtype=torch.long)
        mask=data['mask'].to(device,dtype=torch.long)
        targets=data['targets'].to(device,dtype=torch.long)

        
        outputs=model(ids,mask)
        
        loss=loss_function(outputs,targets)
        tr_loss+=loss.item()
        big_val,big_idx=torch.max(outputs.data,dim=1)#max logit or the softmax layer
        n_correct+=calculate_accu(big_val,target)

        nb_tr_steps+=1
        nb_tr_examples+=targets.size(0)

        if _ %5000==0:
            loss_step=tr_loss/nb_tr_steps
            accu_step=(n_correct*100)/nb_tr_examples
            print(f"Training loss per 5000 steps: {loss_step}")
            print(f"Training accuracy per 5000 steps: {accu_step}")



        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    epoch_accu=(n_correct*100)/nb_tr_examples
    epoch_loss=tr_loss/nb_tr_steps
    print(f"The total accuracy of epoch {epoch} is : {epoch_accu}") 
    print(f"Training loss of epoch: {epoch_loss}")

    return


def validation(epoch,model,testing_loader,loss_function):
    model.eval()
    tr_loss=0
    nb_tr_steps=0
    n_correct=0
    nb_tr_examples=0

    with torch.no_grad()#Sets the gradiants as 0 as we do not calculate gradints during evaluation

    for _,data in enumerate(testing_loader,0):

        ids=data["ids"].to(device,dtype=torch.long)
        mask=data["mask"].to(device,dtype=torch.long)
        targets=data["targets"].to(device,dtype=torch.long)

        outputs=model(ids,mask).squeeze()

        loss=loss_function(outputs,target)
        tr_loss+=loss.item()
        big_val,big_idx=torch.max(output.data,dim=1)
        n_correct+=calculate_accu(big_val,target)

        nb_tr_steps+=1
        nb_tr_examples+=targets.size(0)

        if _ %1000==0:
            loss_step=tr_loss/nb_tr_steps
            accu_step=(n_correct*100)/nb_tr_examples
            print(f"Validation loss per 1000 steps: {loss_step}")
            print(f"Validation accuracy per 1000 steps: {accu_step}")


        epoch_accu=(n_correct*100)/nb_tr_examples
        epoch_loss=tr_loss/nb_tr_steps
        print(f"The total validation accuracy of epoch {epoch} is : {epoch_accu}") 
        print(f"Validation loss of epoch: {epoch_loss}")

    return 


def main():
    print("-------START---------")
    parser=argparse.ArgumentParser()
    parser.add_argument("--epochs",type=int,default=10)
    parser.add_argument("--learning_rate",type=int,default=1e-05)
    parser.add_argument("--train_batch_size",type=int,default=4)
    parser.add_argument("--valid_batch_size",type=int,default=2)

    args=parser.parse_args()

    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer=DistilBertTokenizer.from_pretrained('distilbert-base-uncase')
    model=DistilBertModel()
    model.to(device)
    LEARNING_RATE=1e-05

    optimizer=torch.optim.Adam(params=model.parameters(),args.learning_rate)
    loss_function=torch.nn.CossEntropy()

    EPOCHS=args.epochs

    for epoch in EPOCHS:
        print(f"Started epoch {epoch}")
        train(epoch,model,training_loader,optimizer,loss_function)
        validation(epoch,model,testing_loader,loss_function)


    output_dir=os.environ["SM_MODEL_DIR"]

    output_model_file=os.path.join(output_dir,"pytorch_distilbert_news.bin")
    output_vocab_file=os.path.join(output_dir,"vocab_distilbert_news.bin")

    torch.save(model.state_dict(),output_model_file)
    tokenizer.save_vocabulary(output_vocab_file)

if __name__=='__main__':
    main()
    
    
        
    


    
    
    

        
        











ModuleNotFoundError: No module named 'sagemaker'